# 02 — Feature Engineering

**Goal:** Build the master feature table by merging all Home Credit tables
and deriving behavioral, demographic, and credit bureau features.

## Contents
1. Load all raw tables
2. Application-level feature engineering
3. Installment payment behavioral features
4. Credit card balance features
5. Bureau history features
6. Previous application features
7. Merge into master table
8. Feature importance preview (mutual information)

In [1]:
import sys, warnings, os
from pathlib import Path

_root = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path().resolve().parent
sys.path.insert(0, str(_root / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

from features import (
    build_application_features, build_bureau_features,
    build_previous_app_features, build_installment_features,
    build_cc_features, build_pos_features,
)
from utils.config import load_config  # side-effect: os.chdir(project_root)

sns.set_theme(style='whitegrid')
cfg = load_config()
print('CWD:', os.getcwd())


CWD: C:\Users\BIPLOB GON\Google Drive\DS & Analytics\github_projects\2026\proactive-defaulter-flagging-system


## 1. Load Raw Tables

In [2]:

# ── Memory-efficient CSV loader ───────────────────────────────────────────────
# DEV_MODE: set True to read only a fraction of each table (faster / less RAM).
# Set False (or unset) for a full production run.
DEV_MODE   = True
SAMPLE_N   = 50_000   # rows from application_train when DEV_MODE is True

def _read(path, **kwargs):
    """Read CSV with dtype downcasting to reduce memory footprint."""
    df = pd.read_csv(path, low_memory=False, **kwargs)
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        try:
            df[col] = pd.to_numeric(df[col], downcast='integer')
        except Exception:
            pass
    return df

def _read_filtered(path, id_col, id_set):
    """Stream-read CSV retaining only rows whose `id_col` is in `id_set`."""
    chunks = []
    for chunk in pd.read_csv(path, chunksize=200_000, low_memory=False):
        match = chunk[chunk[id_col].isin(id_set)].copy()
        for col in match.select_dtypes('float64').columns:
            match[col] = match[col].astype('float32')
        if not match.empty:
            chunks.append(match)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

if DEV_MODE:
    app   = _read(cfg.data.application_train, nrows=SAMPLE_N)
    curr_ids   = set(app['SK_ID_CURR'])

    bur   = _read_filtered(cfg.data.bureau, 'SK_ID_CURR', curr_ids)
    bur_ids    = set(bur['SK_ID_BUREAU']) if 'SK_ID_BUREAU' in bur.columns else set()
    bb    = _read_filtered(cfg.data.bureau_balance, 'SK_ID_BUREAU', bur_ids)

    prev  = _read_filtered(cfg.data.previous_application,  'SK_ID_CURR', curr_ids)
    inst  = _read_filtered(cfg.data.installments_payments,  'SK_ID_CURR', curr_ids)
    cc    = _read_filtered(cfg.data.credit_card_balance,    'SK_ID_CURR', curr_ids)
    pos   = _read_filtered(cfg.data.pos_cash_balance,       'SK_ID_CURR', curr_ids)
else:
    app  = _read(cfg.data.application_train)
    bur  = _read(cfg.data.bureau)
    bb   = _read(cfg.data.bureau_balance)
    prev = _read(cfg.data.previous_application)
    inst = _read(cfg.data.installments_payments)
    cc   = _read(cfg.data.credit_card_balance)
    pos  = _read(cfg.data.pos_cash_balance)

print(f'DEV_MODE={DEV_MODE}  — Tables loaded:')
for name, df in [('app',app),('bureau',bur),('bureau_bal',bb),
                  ('prev',prev),('inst',inst),('cc',cc),('pos',pos)]:
    print(f'  {name}: {df.shape}')


DEV_MODE=True  — Tables loaded:
  app: (50000, 122)
  bureau: (238223, 17)
  bureau_bal: (2376827, 3)
  prev: (229287, 37)
  inst: (1868928, 8)
  cc: (521346, 23)
  pos: (1380787, 8)


## 2. Application Features

In [3]:
app_feats = build_application_features(app)
new_cols = set(app_feats.columns) - set(app.columns)
print(f'New features added: {sorted(new_cols)}')
app_feats[list(new_cols)[:6]].head(3)

2026-04-06 00:06:12 | INFO     | features.applicant_features | Building application features for 50000 rows ...
2026-04-06 00:06:12 | INFO     | features.applicant_features | Application features built. Final shape: (50000, 132)
New features added: ['age_years', 'annuity_income_ratio', 'credit_goods_ratio', 'credit_income_ratio', 'employed_years', 'employment_ratio', 'ext_source_mean', 'ext_source_min', 'income_per_person', 'n_docs_provided']


,ext_source_mean,n_docs_provided,employment_ratio,ext_source_min,credit_income_ratio,income_per_person
0,0.161787,1,0.067329,0.083037,2.007889,202500.0
1,0.466757,1,0.070862,0.311267,4.790750,135000.0
2,0.642739,0,0.011814,0.555912,2.000000,67500.0


## 3. Behavioral Features — Installment Payments

These capture *how consistently* borrowers make their payments, including:
- Days late (mean/max/std)
- DPD30 / DPD60 / DPD90 flags
- On-time payment velocity

In [ ]:
import importlib
import features.behavioral_features as _bf
importlib.reload(_bf)
from features.behavioral_features import build_installment_features, build_cc_features

inst_feats = build_installment_features(inst)
print('Installment features:', inst_feats.shape)

# Visualise DPD30 rate distribution by default status
temp = inst_feats.merge(app[['SK_ID_CURR', 'TARGET']], on='SK_ID_CURR')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t, color, label in [(0,'#2ecc71','Non-Default'), (1,'#e74c3c','Default')]:
    axes[0].hist(temp.loc[temp['TARGET']==t, 'inst_days_late_mean'].clip(-30,60),
                  bins=50, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Mean Days Late Distribution', fontweight='bold')
axes[0].legend()

axes[1].bar(['Non-Default','Default'],
            [temp.loc[temp['TARGET']==0,'inst_dpd30_rate'].mean(),
             temp.loc[temp['TARGET']==1,'inst_dpd30_rate'].mean()],
            color=['#2ecc71','#e74c3c'])
axes[1].set_title('Average DPD30 Rate', fontweight='bold')
axes[1].set_ylabel('Rate')
plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/02_behavioral_features.png', dpi=150, bbox_inches='tight')
plt.show()


2026-04-06 00:07:16 | INFO     | features.behavioral_features | Building installment payment features for 1868928 records ...


KeyError: 'DAYS_PAID_INSTALMENT'

## 4. Bureau & Previous Application Features

In [ ]:
bureau_feats = build_bureau_features(bur, bb)
prev_feats   = build_previous_app_features(prev)
cc_feats     = build_cc_features(cc)
pos_feats    = build_pos_features(pos)

print('Bureau features:',   bureau_feats.shape)
print('Previous app feats:',prev_feats.shape)
print('CC balance feats:',  cc_feats.shape)
print('POS cash feats:',    pos_feats.shape)

## 5. Merge into Master Table

In [ ]:
master = (
    app_feats
    .merge(bureau_feats,  on='SK_ID_CURR', how='left')
    .merge(prev_feats,    on='SK_ID_CURR', how='left')
    .merge(inst_feats,    on='SK_ID_CURR', how='left')
    .merge(cc_feats,      on='SK_ID_CURR', how='left')
    .merge(pos_feats,     on='SK_ID_CURR', how='left')
)

print(f'Master table: {master.shape}')
print(f'Default rate preserved: {master["TARGET"].mean():.2%}')
master.head(3)

## 6. Feature Importance Preview (Mutual Information)

In [ ]:
# Numeric features only, drop high-missing
numeric = master.select_dtypes(include='number').copy()
numeric = numeric.drop(columns=['SK_ID_CURR'], errors='ignore')
numeric = numeric.loc[:, numeric.isnull().mean() < 0.5]
numeric = numeric.fillna(numeric.median())

y = numeric.pop('TARGET')
mi = mutual_info_classif(numeric, y, random_state=42)
mi_series = pd.Series(mi, index=numeric.columns).sort_values(ascending=False)

top20 = mi_series.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1],
        color=sns.color_palette('Blues_d', len(top20)))
ax.set_xlabel('Mutual Information Score')
ax.set_title('Top 20 Features by Mutual Information', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/02_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()

# Save master
os.makedirs('data/processed/feature_cache', exist_ok=True)
master.to_csv('data/processed/feature_cache/master_features.csv', index=False)
print('Master features saved.')
